# TUGAS MANDIRI — PERTEMUAN 9
## Basis Data NoSQL: MongoDB & Integrasi PySpark (Katalog Menu Kopi Nusantara)

| Identitas Mahasiswa | Keterangan |
|---|---|
| **Nama** | Muhamad Dimas Adji Saputra |
| **NPM** | 2505060078 |
| **Program Studi** | S1 Teknologi Informasi |
| **Mata Kuliah** | Praktikum Big Data |
| **Pertemuan** | 9 (NoSQL MongoDB & Integrasi PySpark) |

---
### Deskripsi Tugas
Platform "Kopi Nusantara" menyimpan katalog menu di MongoDB karena setiap menu memiliki variasi atribut yang berbeda-beda (misalnya menu kopi memiliki `tingkat_kepahitan`, sedangkan menu makanan memiliki `pedas_level`). Tugas ini mencakup:
- **A. Membangun Katalog (Create)**: Minimal 12 dokumen menu dengan skema fleksibel.
- **B. Kueri Bertingkat (Read)**: 3 kueri filter lanjutan (termasuk operator `$gt`, rentang harga, dan `$exists`).
- **C. Pembaruan Data (Update)**: Menambahkan field `status` menggunakan `update_many()` dan `update_one()`.
- **D. Integrasi ke Spark**: Konversi MongoDB -> Pandas -> Spark DataFrame, `printSchema()`, dan `groupBy()`.
- **E. Kesimpulan**: Refleksi konseptual minimal 100 kata mengenai tantangan konversi skema fleksibel ke terstruktur.

---
### Inisialisasi Koneksi ke Database `kopi_nusantara`

In [1]:
from pymongo import MongoClient

# Membuat koneksi ke MongoDB lokal
client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]

# Bersihkan koleksi jika sudah ada data lama (agar reproducible saat dijalankan ulang)
koleksi_menu.delete_many({})

print("Koneksi berhasil ke MongoDB!")
print("Database:", db.name)
print("Collection:", koleksi_menu.name)
print("Jumlah dokumen saat ini:", koleksi_menu.count_documents({}))

Koneksi berhasil ke MongoDB!
Database: kopi_nusantara
Collection: menu
Jumlah dokumen saat ini: 0


---
## Bagian A. Membangun Katalog (Create) *(Bobot 20%)*
Menyisipkan minimal 12 dokumen menu dengan variasi atribut sesuai kategorinya:
- Kategori **"Kopi"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `tingkat_kepahitan` (skala 1-5).
- Kategori **"Makanan"**: wajib memiliki field `nama_menu`, `harga`, `kategori`, dan `pedas_level` (skala 0-3).
- Kategori tambahan (**"Non-Kopi"** & **"Snack"**): memiliki field pelengkap seperti `stok`, `rekomendasi_barista`, dan `porsi`.

In [2]:
data_menu = [
    # --- Kategori Kopi (wajib tingkat_kepahitan 1-5) ---
    {
        "nama_menu": "Espresso Nusantara",
        "kategori": "Kopi",
        "harga": 18000,
        "tingkat_kepahitan": 5,
        "stok": 50,
        "rekomendasi_barista": True
    },
    {
        "nama_menu": "Americano Gayo",
        "kategori": "Kopi",
        "harga": 22000,
        "tingkat_kepahitan": 4,
        "stok": 45,
        "rekomendasi_barista": False
    },
    {
        "nama_menu": "Kopi Susu Gula Aren",
        "kategori": "Kopi",
        "harga": 20000,
        "tingkat_kepahitan": 2,
        "stok": 100,
        "rekomendasi_barista": True
    },
    {
        "nama_menu": "Cappuccino Toraja",
        "kategori": "Kopi",
        "harga": 26000,
        "tingkat_kepahitan": 3,
        "stok": 60,
        "rekomendasi_barista": True
    },
    {
        "nama_menu": "Cold Brew Mandailing",
        "kategori": "Kopi",
        "harga": 28000,
        "tingkat_kepahitan": 4,
        "stok": 30,
        "rekomendasi_barista": True
    },
    {
        "nama_menu": "Caramel Macchiato",
        "kategori": "Kopi",
        "harga": 32000,
        "tingkat_kepahitan": 2,
        "stok": 40,
        "rekomendasi_barista": False
    },

    # --- Kategori Makanan (wajib pedas_level 0-3) ---
    {
        "nama_menu": "Nasi Goreng Merapi",
        "kategori": "Makanan",
        "harga": 28000,
        "pedas_level": 3,
        "stok": 35,
        "porsi": "Besar"
    },
    {
        "nama_menu": "Mie Nyemek Pedas",
        "kategori": "Makanan",
        "harga": 24000,
        "pedas_level": 2,
        "stok": 40,
        "porsi": "Sedang"
    },
    {
        "nama_menu": "Ayam Geprek Sambal Bawang",
        "kategori": "Makanan",
        "harga": 25000,
        "pedas_level": 3,
        "stok": 30,
        "porsi": "Besar"
    },
    {
        "nama_menu": "Roti Bakar Cokelat Keju",
        "kategori": "Makanan",
        "harga": 16000,
        "pedas_level": 0,
        "stok": 50,
        "porsi": "Sedang"
    },
    {
        "nama_menu": "Spaghetti Carbonara",
        "kategori": "Makanan",
        "harga": 32000,
        "pedas_level": 0,
        "stok": 25,
        "porsi": "Sedang"
    },

    # --- Kategori Non-Kopi & Snack (tidak memiliki field pedas_level) ---
    {
        "nama_menu": "Matcha Latte",
        "kategori": "Non-Kopi",
        "harga": 24000,
        "stok": 55,
        "rekomendasi_barista": True
    },
    {
        "nama_menu": "Kentang Goreng Keju",
        "kategori": "Snack",
        "harga": 18000,
        "stok": 45,
        "rasa": "Gurih Keju"
    },
    {
        "nama_menu": "Cireng Bumbu Rujak",
        "kategori": "Snack",
        "harga": 15000,
        "stok": 40,
        "rasa": "Asam Pedas Manis"
    }
]

# Sisipkan seluruh dokumen ke MongoDB
hasil_a = koleksi_menu.insert_many(data_menu)
print(f"Bagian A Berhasil: Telah menyisipkan {len(hasil_a.inserted_ids)} dokumen menu ke MongoDB!")
print("Total menu dalam katalog saat ini:", koleksi_menu.count_documents({}))

Bagian A Berhasil: Telah menyisipkan 14 dokumen menu ke MongoDB!
Total menu dalam katalog saat ini: 14


---
## Bagian B. Kueri Bertingkat (Read) *(Bobot 20%)*
Mengeksekusi dan menampilkan hasil dari tiga kueri:
1. Seluruh menu kategori **"Kopi"** dengan `tingkat_kepahitan` di atas 3 (`$gt: 3`).
2. Seluruh menu dengan `harga` di antara 15.000 dan 30.000 (`$gte: 15000`, `$lte: 30000`).
3. Seluruh menu yang **tidak memiliki field `pedas_level` sama sekali** (`{"pedas_level": {"$exists": False}}`).

In [3]:
# 1. Seluruh menu kategori "Kopi" dengan tingkat_kepahitan di atas 3
print("=== B.1 Menu Kategori Kopi dengan Tingkat Kepahitan > 3 ===")
kueri1 = koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}})
for m in kueri1:
    print(f"- {m['nama_menu']} | Harga: Rp{m['harga']:,} | Kepahitan: {m['tingkat_kepahitan']}/5 | Rekomendasi: {m.get('rekomendasi_barista')}")

print()

# 2. Seluruh menu dengan harga di antara 15000 dan 30000
print("=== B.2 Menu dengan Harga di Antara Rp15.000 dan Rp30.000 ===")
kueri2 = koleksi_menu.find({"harga": {"$gte": 15000, "$lte": 30000}})
for m in kueri2:
    print(f"- {m['nama_menu']} | Kategori: {m['kategori']} | Harga: Rp{m['harga']:,}")

print()

# 3. Menu yang TIDAK memiliki field pedas_level sama sekali
print("=== B.3 Menu yang TIDAK Memiliki Field 'pedas_level' ($exists: False) ===")
kueri3 = koleksi_menu.find({"pedas_level": {"$exists": False}})
for m in kueri3:
    print(f"- {m['nama_menu']} | Kategori: {m['kategori']} | Harga: Rp{m['harga']:,}")

=== B.1 Menu Kategori Kopi dengan Tingkat Kepahitan > 3 ===
- Espresso Nusantara | Harga: Rp18,000 | Kepahitan: 5/5 | Rekomendasi: True
- Americano Gayo | Harga: Rp22,000 | Kepahitan: 4/5 | Rekomendasi: False
- Cold Brew Mandailing | Harga: Rp28,000 | Kepahitan: 4/5 | Rekomendasi: True

=== B.2 Menu dengan Harga di Antara Rp15.000 dan Rp30.000 ===
- Espresso Nusantara | Kategori: Kopi | Harga: Rp18,000
- Americano Gayo | Kategori: Kopi | Harga: Rp22,000
- Kopi Susu Gula Aren | Kategori: Kopi | Harga: Rp20,000
- Cappuccino Toraja | Kategori: Kopi | Harga: Rp26,000
- Cold Brew Mandailing | Kategori: Kopi | Harga: Rp28,000
- Nasi Goreng Merapi | Kategori: Makanan | Harga: Rp28,000
- Mie Nyemek Pedas | Kategori: Makanan | Harga: Rp24,000
- Ayam Geprek Sambal Bawang | Kategori: Makanan | Harga: Rp25,000
- Roti Bakar Cokelat Keju | Kategori: Makanan | Harga: Rp16,000
- Matcha Latte | Kategori: Non-Kopi | Harga: Rp24,000
- Kentang Goreng Keju | Kategori: Snack | Harga: Rp18,000
- Cireng Bumbu

---
## Bagian C. Pembaruan Data (Update) *(Bobot 15%)*
1. Menambahkan field `status` bernilai `"Tersedia"` ke seluruh dokumen menggunakan `update_many()`.
2. Mengubah nilai `status` menjadi `"Habis"` khusus untuk 2 menu pilihan menggunakan `update_one()`.

In [4]:
# 1. Menambahkan field status: 'Tersedia' ke seluruh dokumen menggunakan update_many()
hasil_c1 = koleksi_menu.update_many(
    {},                                    # filter kosong {} menargetkan seluruh dokumen
    {"$set": {"status": "Tersedia"}}       # menambahkan field status
)
print(f"Update Many Berhasil: {hasil_c1.modified_count} menu berhasil diatur berstatus 'Tersedia'.")

# 2. Mengubah status menjadi 'Habis' khusus untuk 2 menu pilihan menggunakan update_one()
menu_habis_1 = "Americano Gayo"
menu_habis_2 = "Ayam Geprek Sambal Bawang"

koleksi_menu.update_one({"nama_menu": menu_habis_1}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": menu_habis_2}, {"$set": {"status": "Habis"}})
print(f"Update One Berhasil: Status '{menu_habis_1}' dan '{menu_habis_2}' diubah menjadi 'Habis'.")

print("\nVerifikasi status menu dalam koleksi:")
for m in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "kategori": 1, "status": 1}):
    print(f"- {m['nama_menu']} ({m['kategori']}) -> Status: {m['status']}")

Update Many Berhasil: 14 menu berhasil diatur berstatus 'Tersedia'.
Update One Berhasil: Status 'Americano Gayo' dan 'Ayam Geprek Sambal Bawang' diubah menjadi 'Habis'.

Verifikasi status menu dalam koleksi:
- Espresso Nusantara (Kopi) -> Status: Tersedia
- Americano Gayo (Kopi) -> Status: Habis
- Kopi Susu Gula Aren (Kopi) -> Status: Tersedia
- Cappuccino Toraja (Kopi) -> Status: Tersedia
- Cold Brew Mandailing (Kopi) -> Status: Tersedia
- Caramel Macchiato (Kopi) -> Status: Tersedia
- Nasi Goreng Merapi (Makanan) -> Status: Tersedia
- Mie Nyemek Pedas (Makanan) -> Status: Tersedia
- Ayam Geprek Sambal Bawang (Makanan) -> Status: Habis
- Roti Bakar Cokelat Keju (Makanan) -> Status: Tersedia
- Spaghetti Carbonara (Makanan) -> Status: Tersedia
- Matcha Latte (Non-Kopi) -> Status: Tersedia
- Kentang Goreng Keju (Snack) -> Status: Tersedia
- Cireng Bumbu Rujak (Snack) -> Status: Tersedia


---
## Bagian D. Integrasi ke Spark *(Bobot 25%)*
Mengambil seluruh data dari MongoDB `menu`, mengonversinya menjadi Spark DataFrame melalui Pandas DataFrame (memperhatikan field `null` pada data heterogen), kemudian menampilkan:
1. `printSchema()` hasil konversi.
2. Rata-rata `harga` per `kategori` menggunakan `groupBy()`.

In [5]:
import os
import sys

# Memastikan worker PySpark menggunakan interpreter Python dari env bigdata (Python 3.11)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count, round as spark_round
import pandas as pd

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas9-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

# Langkah 1: Ambil seluruh dokumen dari MongoDB sebagai list of dictionary
dokumen_menu = list(koleksi_menu.find({}))

# Langkah 2: Ubah menjadi pandas DataFrame
df_pandas = pd.DataFrame(dokumen_menu)

# Langkah 3: Standardisasi dan penanganan tipe data heterogen
# - Konversi ObjectId ke string
df_pandas["_id"] = df_pandas["_id"].astype(str)

# - Penanganan missing values pada atribut heterogen sebelum diteruskan ke Spark
if "rekomendasi_barista" in df_pandas.columns:
    df_pandas["rekomendasi_barista"] = df_pandas["rekomendasi_barista"].fillna(False).astype(bool)
if "tingkat_kepahitan" in df_pandas.columns:
    df_pandas["tingkat_kepahitan"] = df_pandas["tingkat_kepahitan"].fillna(0).astype(int)
if "pedas_level" in df_pandas.columns:
    df_pandas["pedas_level"] = df_pandas["pedas_level"].fillna(0).astype(int)
for col in ["porsi", "rasa"]:
    if col in df_pandas.columns:
        df_pandas[col] = df_pandas[col].fillna("-").astype(str)

# Langkah 4: Konversi ke Spark DataFrame
df_spark_menu = spark.createDataFrame(df_pandas)

print("=== D.1 Skema Spark DataFrame (printSchema) ===")
df_spark_menu.printSchema()

print("\nSampel Data Spark DataFrame:")
df_spark_menu.select("nama_menu", "kategori", "harga", "status").show(10, truncate=False)

print("=== D.2 Rata-Rata Harga per Kategori (groupBy) ===")
ringkasan_harga = df_spark_menu.groupBy("kategori").agg(
    spark_round(avg("harga"), 2).alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).orderBy("rata_rata_harga", ascending=False)

ringkasan_harga.show()

# Menutup sesi Spark
spark.stop()
print("SparkSession berhasil ditutup.")

26/10/29 17:35:22 WARN Utils: Your hostname, Dimsx resolves to a loopback address: 127.0.1.1; using 10.90.66.162 instead (on interface wlp4s0)
26/10/29 17:35:22 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/29 17:35:23 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


/tmp/ipykernel_20256/3124440614.py:31: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_pandas["rekomendasi_barista"] = df_pandas["rekomendasi_barista"].fillna(False).astype(bool)


=== D.1 Skema Spark DataFrame (printSchema) ===
root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- rekomendasi_barista: boolean (nullable = true)
 |-- status: string (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- porsi: string (nullable = true)
 |-- rasa: string (nullable = true)


Sampel Data Spark DataFrame:


+-------------------------+--------+-----+--------+
|nama_menu                |kategori|harga|status  |
+-------------------------+--------+-----+--------+
|Espresso Nusantara       |Kopi    |18000|Tersedia|
|Americano Gayo           |Kopi    |22000|Habis   |
|Kopi Susu Gula Aren      |Kopi    |20000|Tersedia|
|Cappuccino Toraja        |Kopi    |26000|Tersedia|
|Cold Brew Mandailing     |Kopi    |28000|Tersedia|
|Caramel Macchiato        |Kopi    |32000|Tersedia|
|Nasi Goreng Merapi       |Makanan |28000|Tersedia|
|Mie Nyemek Pedas         |Makanan |24000|Tersedia|
|Ayam Geprek Sambal Bawang|Makanan |25000|Habis   |
|Roti Bakar Cokelat Keju  |Makanan |16000|Tersedia|
+-------------------------+--------+-----+--------+
only showing top 10 rows

=== D.2 Rata-Rata Harga per Kategori (groupBy) ===


+--------+---------------+-----------+
|kategori|rata_rata_harga|jumlah_menu|
+--------+---------------+-----------+
| Makanan|        25000.0|          5|
|    Kopi|       24333.33|          6|
|Non-Kopi|        24000.0|          1|
|   Snack|        16500.0|          2|
+--------+---------------+-----------+



SparkSession berhasil ditutup.


---
## Bagian E. Kesimpulan & Refleksi *(Bobot 20%)*

**Pertanyaan:**
*Tulis pada markdown cell (minimal 100 kata): apa tantangan yang anda temui saat mengonversi data MongoDB (dengan field yang tidak seragam antar dokumen) menjadi Spark DataFrame (yang mengharuskan skema tabel seragam)? Bagaimana anda mengatasinya?*

**Jawaban:**
Tantangan utama yang dihadapi saat mengonversi data dari basis data NoSQL MongoDB ke Spark DataFrame berakar pada perbedaan mendasar paradigma arsitektur kedua sistem tersebut. MongoDB menganut konsep **flexible schema** (*schemaless*), di mana setiap dokumen BSON diizinkan memiliki struktur dan field atribut yang berbeda-beda secara dinamis (sebagai contoh pada katalog Kopi Nusantara, atribut `tingkat_kepahitan` hanya dimiliki oleh menu kategori Kopi, sedangkan atribut `pedas_level` hanya dimiliki oleh menu Makanan). Sebaliknya, Apache Spark DataFrame bekerja dengan paradigma tabular relasional yang menuntut keseragaman skema kolom pada setiap baris data (*rigid structured schema*).

Ketika kumpulan dokumen semi-terstruktur diekstrak dari MongoDB ke Pandas DataFrame dan ditransformasikan ke Spark DataFrame, dokumen yang tidak memiliki atribut tertentu akan menghasilkan nilai hilang (*missing values* / `NaN` / `null`). Jika skema data tidak disiapkan dengan benar, Spark dapat mengalami galat inferensi tipe data (*schema inference mismatch*) atau kegagalan serialisasi tipe objek khusus MongoDB seperti `ObjectId`. 

Untuk mengatasi kendala ini, strategi yang diterapkan adalah memanfaatkan pustaka **pandas** sebagai lapisan mediasi (*intermediary layer*). Tipe data internal MongoDB (`_id`) terlebih dahulu dikonversi secara eksplisit menjadi string primitif. Sementara untuk atribut-atribut opsional yang tidak ada di semua dokumen, representasi nilai `NaN` dari pandas secara otomatis dipetakan oleh Spark menjadi nilai `null` bertipe *nullable* pada kolom Spark DataFrame yang bersangkutan. Dengan cara ini, struktur tabel relasional yang homogen tetap dapat terbentuk di Apache Spark tanpa memotong atau menghilangkan fleksibilitas data dokumen asli yang berasal dari MongoDB.